# Notebook 8 — Normalized $R_t$ and Signed Log-Ratio (Item B)

**No retraining required.** Both metrics Mahapatra asked for are pure
functions of the ratio already stored in every `rt_log`, plus two fixed
parameter counts:

$$R_t^{\mathrm{norm}} = \frac{\lVert g_v\rVert/\sqrt{N_v}}{\lVert g_\ell\rVert/\sqrt{N_\ell}}
= \frac{1}{r_t}\sqrt{\frac{N_\ell}{N_v}}, \qquad B_t = \log R_t^{\mathrm{norm}}$$

where $r_t = \lVert g_\ell\rVert / \lVert g_v\rVert$ is the logged value.
Since $N_v, N_\ell$ are constant, normalisation is a constant rescaling:
it will not change any ranking, but it does relocate the balance point,
which is the actual issue -- the paper currently claims $R_t=1$ means
balanced, which is false when the groups differ in size.

**Runtime:** ~2 minutes. Loads BLIP once for parameter counts, then
reprocesses existing JSONs.

**Also audits** the other Item-B disclosure questions (scaled vs
unscaled gradients, module membership, `grad=None` handling, optimizer
state at stage boundary) so the Methods section can state them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'transformers>=4.44,<4.50', '-q'], check=True)
print('ready')

## 1. Parameter counts under both conventions

Group membership must match exactly what `GradientTracker` hooked:
visual = parameter names containing `vision_model`, language = names
containing `text_decoder`. Anything matching neither was never counted.

In [ ]:
import torch, math, json
import numpy as np
from pathlib import Path
from transformers import BlipForConditionalGeneration

model = BlipForConditionalGeneration.from_pretrained(
    'Salesforce/blip-image-captioning-base')

vis_tensors = lang_tensors = other_tensors = 0
vis_elems   = lang_elems   = other_elems   = 0
other_names = []

for name, p in model.named_parameters():
    n = p.numel()
    if 'vision_model' in name:
        vis_tensors += 1;  vis_elems += n
    elif 'text_decoder' in name:
        lang_tensors += 1; lang_elems += n
    else:
        other_tensors += 1; other_elems += n
        other_names.append(name)

print('GROUP MEMBERSHIP (as GradientTracker defines it)')
print('=' * 60)
print(f'visual   : {vis_tensors:4d} tensors, {vis_elems:>12,} elements')
print(f'language : {lang_tensors:4d} tensors, {lang_elems:>12,} elements')
print(f'neither  : {other_tensors:4d} tensors, {other_elems:>12,} elements')
if other_names:
    print('\nParameters in NEITHER group (never entered Rt at all):')
    for nm in other_names[:20]:
        print(f'   {nm}')
    if len(other_names) > 20:
        print(f'   ... and {len(other_names)-20} more')

# Where does cross-attention live? It is the fusion point, so its
# placement is a genuine interpretive question for the paper.
xattn = [n for n, _ in model.named_parameters() if 'crossattention' in n]
print(f'\nCross-attention tensors: {len(xattn)}')
if xattn:
    in_lang = sum(1 for n in xattn if 'text_decoder' in n)
    print(f'   ...of which counted as LANGUAGE: {in_lang}')
    print(f'   example: {xattn[0]}')

K_TENSOR = math.sqrt(lang_tensors / vis_tensors)
K_ELEM   = math.sqrt(lang_elems   / vis_elems)
print('\nNormalisation constants  Rt_norm = (1/r) * K')
print(f'   K (tensor-count convention) = {K_TENSOR:.4f}')
print(f'   K (scalar-element convention) = {K_ELEM:.4f}')
print(f'\nBalance point (Rt_norm = 1) occurs at logged ratio r = K:')
print(f'   tensor convention : r = {K_TENSOR:.4f}')
print(f'   element convention: r = {K_ELEM:.4f}')
print('\n=> Any statement that "r = 1 means balanced" is incorrect.')

## 2. Recompute $R_t^{\mathrm{norm}}$ and $B_t$ for every logged run

In [ ]:
OUT = Path('/content/drive/MyDrive/DAMF/logs')
CONV = 'element'          # 'element' or 'tensor'
K = K_ELEM if CONV == 'element' else K_TENSOR
print(f'Using {CONV} convention, K = {K:.4f}\n')

METHODS = ['naive_ft', 'lowlr_ft', 'damf',
           'joint_schedule_ft', 'rms_balanced_ft']
DATASETS = ['uicd', 'rsicd', 'rocov2']
SEEDS = [42, 0, 123]

def window_stats(vals, n=10):
    v = np.asarray(vals, dtype=float)
    k = min(n, len(v))
    return float(v[:k].mean()), float(v[-k:].mean())

rows = []
for method in METHODS:
    for ds in DATASETS:
        e_raw, l_raw, e_nrm, l_nrm, e_b, l_b = [], [], [], [], [], []
        for seed in SEEDS:
            f = OUT / f'{method}_{ds}_seed{seed}.json'
            if not f.exists():
                continue
            rt = json.load(open(f)).get('rt_log', [])
            if not rt:
                continue
            r = np.array([x[1] for x in rt], dtype=float)
            r = r[r > 0]
            if r.size == 0:
                continue
            rn = (1.0 / r) * K
            b  = np.log(rn)
            a, z = window_stats(r);  e_raw.append(a); l_raw.append(z)
            a, z = window_stats(rn); e_nrm.append(a); l_nrm.append(z)
            a, z = window_stats(b);  e_b.append(a);   l_b.append(z)
        if not e_raw:
            continue
        rows.append(dict(method=method, dataset=ds, n=len(e_raw),
                         raw_early=np.mean(e_raw), raw_late=np.mean(l_raw),
                         norm_early=np.mean(e_nrm), norm_late=np.mean(l_nrm),
                         b_early=np.mean(e_b), b_late=np.mean(l_b),
                         b_late_sd=(np.std(l_b, ddof=1) if len(l_b) > 1 else 0.0)))

hdr = (f"{'method':18s}{'ds':8s}{'n':>2s}  {'r_early':>8s}{'r_late':>8s}"
       f"{'Rn_early':>10s}{'Rn_late':>9s}{'B_early':>9s}{'B_late':>9s}")
print(hdr); print('-' * len(hdr))
for x in rows:
    print(f"{x['method']:18s}{x['dataset']:8s}{x['n']:2d}  "
          f"{x['raw_early']:8.3f}{x['raw_late']:8.3f}"
          f"{x['norm_early']:10.3f}{x['norm_late']:9.3f}"
          f"{x['b_early']:9.3f}{x['b_late']:9.3f}")

print('\nInterpretation of B_t:')
print('  B_t = 0  -> groups balanced per-parameter')
print('  B_t > 0  -> VISUAL group larger per-parameter gradient')
print('  B_t < 0  -> LANGUAGE group larger per-parameter gradient')

## 3. LaTeX table for the paper

In [ ]:
disp = {'naive_ft': 'Naive FT', 'lowlr_ft': 'Low-LR FT', 'damf': 'Staged FT',
        'joint_schedule_ft': 'Joint-Schedule FT',
        'rms_balanced_ft': 'RMS-Balanced FT'}
dsdisp = {'uicd': 'UICD', 'rsicd': 'RSICD', 'rocov2': 'ROCOv2'}

L = []
L.append(r'\begin{table*}[!htbp]')
L.append(r'\centering')
L.append(r'\caption{Raw ratio $r_t$, parameter-normalised $R_t^{\mathrm{norm}}$ '
         r'and signed log-ratio $B_t$, averaged over the first and last ten '
         r'logged steps. Normalisation uses the scalar-element convention '
         rf'($K={K:.3f}$). $B_t=0$ denotes per-parameter balance; positive '
         r'values indicate a larger visual-side gradient. Note that raw '
         rf'$r_t=1$ does \emph{{not}} denote balance: that point sits at '
         rf'$r_t={K:.3f}$.}}')
L.append(r'\label{tab:normalised-rt}')
L.append(r'\begin{tabular}{llcccccc}')
L.append(r'\toprule')
L.append(r'\textbf{Dataset} & \textbf{Method} & \textbf{$r_t$ early} & '
         r'\textbf{$r_t$ late} & \textbf{$R_t^{\mathrm{norm}}$ early} & '
         r'\textbf{$R_t^{\mathrm{norm}}$ late} & \textbf{$B_t$ early} & '
         r'\textbf{$B_t$ late} \\')
L.append(r'\midrule')
for ds in DATASETS:
    sub = [x for x in rows if x['dataset'] == ds]
    for i, x in enumerate(sub):
        d = dsdisp[ds] if i == 0 else ''
        L.append(f"{d} & {disp[x['method']]} & {x['raw_early']:.2f} & "
                 f"{x['raw_late']:.2f} & {x['norm_early']:.3f} & "
                 f"{x['norm_late']:.3f} & {x['b_early']:+.2f} & "
                 f"{x['b_late']:+.2f} \\\\")
    L.append(r'\midrule')
if L[-1] == r'\midrule':
    L.pop()
L += [r'\bottomrule', r'\end{tabular}', r'\end{table*}']
print('\n'.join(L))

## 4. Measurement-protocol audit (the rest of Item B)

These are disclosure questions, answered by reading the training code
rather than by computation. Recorded here so the Methods section can
state them precisely.

In [ ]:
print("""
ITEM-B MEASUREMENT PROTOCOL AUDIT
=================================

1. SCALED vs UNSCALED GRADIENTS  -- differs between conditions
   GradientTracker registers per-parameter hooks via
   param.register_hook(). These fire during backward(), i.e. on the
   LOSS-SCALED gradient, before scaler.unscale_() is ever called.
   Naive FT / Low-LR FT / Staged FT / Joint-Schedule FT therefore
   report r_t computed on SCALED gradients.
   RMS-Balanced FT instead calls scaler.unscale_(optimizer) and then
   reads .grad directly, so its r_t is computed on UNSCALED gradients.
   Because the AMP loss scale multiplies BOTH groups identically, the
   RATIO is invariant to it -- the two are comparable. Absolute norms
   would not have been.

2. GRADIENT CLIPPING          -- none applied in any condition.
3. GRADIENT ACCUMULATION      -- none; one optimizer step per batch.
4. grad=None HANDLING         -- hooks fire only when a gradient
   exists, and get_rt() returns None if either group's list is empty;
   such steps are skipped, not logged as zero. Non-finite values are
   also filtered before logging.
5. GROUP MEMBERSHIP           -- substring match on parameter names:
   'vision_model' -> visual, 'text_decoder' -> language.
   Consequence: the text decoder's CROSS-ATTENTION blocks are counted
   as LANGUAGE even though they are the fusion point, and the LM head
   sits inside text_decoder. Any parameter matching neither string
   never entered r_t at all (see cell 1 output for the list).
6. OPTIMIZER STATE AT STAGE BOUNDARY
   Staged FT constructs a FRESH AdamW for Stage 2; first- and
   second-moment estimates from Stage 1 are discarded.
   Joint-Schedule FT does the SAME at its phase boundary despite no
   freezing -- which is what makes it a valid control for this
   confound: both reset, only one freezes.
7. LOGGING CADENCE            -- every 10 optimizer steps, all runs.
""")